# Texas production → flare-site panel

Builds the Texas half of the analysis dataset. Aggregates sited RRC
per-well production to site-month, subsets the VIIRS monthly panel to
Texas sites, and outer-joins the two into the analysis panels.

Outputs: `site_month_final.csv`, `texas_sites_full.csv`,
`viirs_monthly_texas.csv`, `viirs_monthly_texas_with_cloud_mask.csv`,
`site_month_viirs_joined_final.csv`,
`site_month_viirs_with_cloud_mask_joined_final.csv`.

## 1. Aggregate to site-month

Read `permian_prod_per_well_with_site_id.csv` and drop wells with
`site_id == -1` (no flare site assigned). Sum every volume column
(`*_bbl`, `*_mcf`) by (`site_id`, `date`) to get one production record
per flare site per month → `site_month_final.csv`.

In [5]:
import pandas as pd

df = pd.read_csv("../../../data/processed/texas/permian_only/permian_prod_per_well_with_site_id.csv")

df = df[df['site_id'] != -1]

value_cols = [c for c in df.columns if c.endswith(("_bbl", "_mcf"))]

site_month = (
    df.groupby(["site_id", "date"])[value_cols]
      .sum()
      .reset_index()
)

### Sanity check: volume conserved

Total across all volume columns is identical before and after the
groupby (21,577,100,447.53), confirming the aggregation neither drops
nor double-counts rows. Note this checks the groupby only — it runs
after the `site_id == -1` drop, so it says nothing about how much
production that filter removed.

In [6]:
print(df[value_cols].sum().sum(), "vs", site_month[value_cols].sum().sum())

21577100447.530098 vs 21577100447.530098


In [7]:
site_month.to_csv("../../../data/processed/merging/site_month_final.csv", index=False)

## 2. Texas site subset

From the full site catalog, keep `state == "TX"` → `texas_sites_full.csv`.
This is the Texas flare-site universe used to filter the VIIRS panel.

In [8]:
df = pd.read_csv("../../../data/processed/nightfire/permian_sites_full.csv")

In [9]:
df = df[df["state"] == "TX"]

In [10]:
df.to_csv("../../../data/processed/nightfire/texas_sites_full.csv", index = False)

## 3. Filter VIIRS monthly panel to Texas sites

Restrict the combined VIIRS monthly output to detections whose `flare_id`
is a Texas site `ID`. Run twice: base panel
(`vnf_sites_aggregated/combined_output.csv` → `viirs_monthly_texas.csv`)
and cloud-mask version
(`vnf_sites_aggregated_with_cloud_mask/combined_output.csv` →
`viirs_monthly_texas_with_cloud_mask.csv`).

In [11]:
big = pd.read_csv("../../../data/processed/nightfire/vnf_sites_aggregated/combined_output.csv")
tx  = pd.read_csv("../../../data/processed/nightfire/texas_sites_full.csv")

tx_ids = set(tx["ID"])                     # the Texas site_id set
result = big[big["flare_id"].isin(tx_ids)]       # keep only rows whose site is in TX

result.to_csv("../../../data/processed/merging/viirs_monthly_texas.csv", index=False)

In [12]:
import pandas as pd
big = pd.read_csv("../../../data/processed/nightfire/vnf_sites_aggregated_with_cloud_mask/combined_output.csv")
tx  = pd.read_csv("../../../data/processed/nightfire/texas_sites_full.csv")

tx_ids = set(tx["ID"])                     # the Texas site_id set
result = big[big["flare_id"].isin(tx_ids)]       # keep only rows whose site is in TX

result.to_csv("../../../data/processed/merging/viirs_monthly_texas_with_cloud_mask.csv", index=False)

## 4. Join production to VIIRS (base panel)

Outer-join `site_month_final.csv` to `viirs_monthly_texas.csv` on
(`site_id`, `year_month`). `flare_id` is renamed to `site_id` (same
identifier); production `site_id` is cast back to int64 after upstream
NaN coercion. Outer join is deliberate — VIIRS-only months (a site was
looked at but had no production record) and production-only months must
both survive → `site_month_viirs_joined_final.csv`.

In [13]:
import pandas as pd

prod = pd.read_csv("../../../data/processed/merging/site_month_final.csv")
viirs = pd.read_csv("../../../data/processed/merging/viirs_monthly_texas.csv")

# production site_id is float64 only from upstream NaN coercion; values are all
# integer-valued. VIIRS flare_id is the SAME identifier, different name.
prod["site_id"] = prod["site_id"].astype("int64")
prod["year_month"] = pd.to_datetime(prod["date"]).dt.strftime("%Y-%m")

viirs = viirs.rename(columns={"flare_id": "site_id"})
viirs["site_id"] = viirs["site_id"].astype("int64")
viirs["year_month"] = viirs["year_month"].astype(str)  # already YYYY-MM

### Merge integrity checks

Assert every input row survives the outer join (matched + left_only =
len(prod); matched + right_only = len(viirs)) and that no column bled
across the boundary. Breakdown of the 321,330 rows: 209,603 both,
92,953 VIIRS-only, 18,774 production-only, spanning 2011-02 to 2026-07.
`looked_no_flare` vs. `detected` splits the satellite-observed rows into
clear-look-but-no-detection and actual flare detections.

In [14]:
m = prod.merge(viirs, on=["site_id", "year_month"], how="outer", indicator=True)

# canonical month-start timestamp for every row (incl. VIIRS-only rows where
# production `date` is null); original `date` left untouched.
m["month"] = pd.to_datetime(m["year_month"] + "-01")
m = m.sort_values(["site_id", "month"]).reset_index(drop=True)
m.shape

(321330, 81)

In [15]:
# every input row must survive an outer join
matched = (m["_merge"] == "both").sum()
assert matched + (m["_merge"] == "left_only").sum()  == len(prod)
assert matched + (m["_merge"] == "right_only").sum() == len(viirs)

# no column bled across the merge boundary
assert m.loc[m["_merge"] == "left_only",  "n_obs"].isna().all()
assert m.loc[m["_merge"] == "right_only", "oil_sold_total_bbl"].isna().all()

print(m["_merge"].value_counts().to_string())
print(m["month"].min().date(), "->", m["month"].max().date())

vp = m["_merge"].isin(["both", "right_only"])
print("looked_no_flare:", ((m.loc[vp,"n_obs"]>0) & (m.loc[vp,"n_detect"]==0)).sum())
print("detected       :", (m.loc[vp,"n_detect"]>0).sum())

_merge
both          209603
right_only     92953
left_only      18774
2011-02-01 -> 2026-07-01
looked_no_flare: 217117
detected       : 85439


In [16]:
m.to_csv("../../../data/processed/merging/site_month_viirs_joined_final.csv", index=False)

## 5. Join production to VIIRS (cloud-mask panel)

Identical join to stage 4, but against
`viirs_monthly_texas_with_cloud_mask.csv`. Wider output (96 vs. 81 cols)
from the cloud-mask fields →
`site_month_viirs_with_cloud_mask_joined_final.csv`. This is the primary
analysis panel.

In [17]:
import pandas as pd

prod = pd.read_csv("../../../data/processed/merging/site_month_final.csv")
viirs = pd.read_csv("../../../data/processed/merging/viirs_monthly_texas_with_cloud_mask.csv")

# production site_id is float64 only from upstream NaN coercion; values are all
# integer-valued. VIIRS flare_id is the SAME identifier, different name.
prod["site_id"] = prod["site_id"].astype("int64")
prod["year_month"] = pd.to_datetime(prod["date"]).dt.strftime("%Y-%m")

viirs = viirs.rename(columns={"flare_id": "site_id"})
viirs["site_id"] = viirs["site_id"].astype("int64")
viirs["year_month"] = viirs["year_month"].astype(str)  # already YYYY-MM

In [18]:
m = prod.merge(viirs, on=["site_id", "year_month"], how="outer", indicator=True)

# canonical month-start timestamp for every row (incl. VIIRS-only rows where
# production `date` is null); original `date` left untouched.
m["month"] = pd.to_datetime(m["year_month"] + "-01")
m = m.sort_values(["site_id", "month"]).reset_index(drop=True)
m.shape

(321330, 96)

In [19]:
# every input row must survive an outer join
matched = (m["_merge"] == "both").sum()
assert matched + (m["_merge"] == "left_only").sum()  == len(prod)
assert matched + (m["_merge"] == "right_only").sum() == len(viirs)

# no column bled across the merge boundary
assert m.loc[m["_merge"] == "left_only",  "n_obs"].isna().all()
assert m.loc[m["_merge"] == "right_only", "oil_sold_total_bbl"].isna().all()

print(m["_merge"].value_counts().to_string())
print(m["month"].min().date(), "->", m["month"].max().date())

vp = m["_merge"].isin(["both", "right_only"])
print("looked_no_flare:", ((m.loc[vp,"n_obs"]>0) & (m.loc[vp,"n_detect"]==0)).sum())
print("detected       :", (m.loc[vp,"n_detect"]>0).sum())

_merge
both          209603
right_only     92953
left_only      18774
2011-02-01 -> 2026-07-01
looked_no_flare: 217117
detected       : 85439


In [20]:
m.to_csv("../../../data/processed/merging/site_month_viirs_with_cloud_mask_joined_final.csv", index=False)